# Fetal Distress Detection from CTG Signals

**End-to-end workflow**: data loading → exploration → preprocessing → feature engineering → model training → evaluation

**Dataset**: CTU-CHB Intrapartum Cardiotocography Database (PhysioNet)  
**Task**: Binary classification — predict fetal distress from FHR + UC signals  
**Model**: 1D CNN-LSTM hybrid (PyTorch)

## 1. Setup and Imports

In [ ]:
import os
import sys
import glob
import json
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from scipy import signal as scipy_signal
from scipy.stats import kurtosis, skew

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, matthews_corrcoef
)

from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

# Import our model
from model import FetalDistressModel

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Paths
RAW_DATA_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'
ARTIFACTS_DIR = 'artifacts'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# List all records
hea_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, '*.hea')))
record_ids = [os.path.basename(f).replace('.hea', '') for f in hea_files]
print(f'Found {len(record_ids)} recordings')
print(f'First 10: {record_ids[:10]}')

## 2. Data Loading and Metadata Extraction

Each `.hea` header file contains the recording metadata plus clinical outcomes (pH, Apgar scores, etc.).

In [ ]:
def parse_header_metadata(hea_path: str) -> dict:
    """
    Parse clinical metadata from a .hea header file.
    Returns a dict with all outcome and descriptor fields.
    """
    meta = {'record_id': os.path.basename(hea_path).replace('.hea', '')}
    
    with open(hea_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('#'):
                continue
            # Remove leading '#' and split
            content = line.lstrip('#').strip()
            if content.startswith('---') or content.startswith('--'):
                continue
            if '!' in content:  # Skip !NotReadyYet! markers
                continue
            
            # Parse key-value pairs
            parts = content.split()
            if len(parts) >= 2:
                # Handle multi-word keys like "Gest. weeks", "I.stage"
                # Find the last item that looks like a value
                value_str = parts[-1]
                key = ' '.join(parts[:-1])
                
                try:
                    value = float(value_str)
                except ValueError:
                    value = value_str
                
                meta[key] = value
    
    return meta

# Parse all headers
metadata_list = []
for hea_file in tqdm(hea_files, desc='Parsing headers'):
    meta = parse_header_metadata(hea_file)
    metadata_list.append(meta)

metadata_df = pd.DataFrame(metadata_list)
print(f'\nMetadata shape: {metadata_df.shape}')
metadata_df.head(10)

In [ ]:
# Quick overview of outcome measures
print('=== Outcome Measures Summary ===')
outcome_cols = ['pH', 'BDecf', 'Apgar1', 'Apgar5']
for col in outcome_cols:
    if col in metadata_df.columns:
        print(f'\n{col}:')
        print(metadata_df[col].describe())
        n_missing = metadata_df[col].isna().sum()
        if n_missing > 0:
            print(f'  Missing: {n_missing}')

## 3. Defining the Target Label: What Counts as "Distressed"?

### Rationale

We define fetal distress using **umbilical cord pH** as the primary outcome:

- **pH < 7.15** → **Distressed** (label = 1)  
- **pH ≥ 7.15** → **Normal** (label = 0)

**Why pH 7.15?**
- Clinical literature considers pH < 7.20 as mild acidosis and pH < 7.10 as severe.  
- pH 7.15 is a middle ground that captures meaningful compromise without restricting the positive class to only extreme cases (which would leave too few samples for training).  
- This threshold is consistent with FIGO guidelines and prior CTG-analysis research.  

We also use a **secondary check**: if pH data is missing, we fall back to **Apgar5 < 7** (five-minute Apgar < 7 indicates compromise).  

Records missing **both** pH and Apgar5 are excluded.

In [ ]:
PH_THRESHOLD = 7.15
APGAR5_THRESHOLD = 7

def assign_label(row):
    """
    Assign distress label.
    Primary: pH < 7.15 → distressed.
    Fallback: if pH is missing, use Apgar5 < 7.
    If both missing: NaN (will be excluded).
    """
    ph = row.get('pH', np.nan)
    apgar5 = row.get('Apgar5', np.nan)
    
    if not np.isnan(ph):
        return 1 if ph < PH_THRESHOLD else 0
    elif not np.isnan(apgar5):
        return 1 if apgar5 < APGAR5_THRESHOLD else 0
    else:
        return np.nan

metadata_df['distress'] = metadata_df.apply(assign_label, axis=1)

# Drop records without a label
n_before = len(metadata_df)
metadata_df = metadata_df.dropna(subset=['distress']).copy()
metadata_df['distress'] = metadata_df['distress'].astype(int)
n_after = len(metadata_df)

print(f'Records with valid labels: {n_after}/{n_before} (dropped {n_before - n_after})')
print(f'\nClass distribution:')
print(metadata_df['distress'].value_counts())
print(f'\nDistress prevalence: {metadata_df["distress"].mean():.1%}')

In [ ]:
# Visualise class distribution and pH distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
metadata_df['distress'].value_counts().plot.bar(ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Normal (0)', 'Distressed (1)'], rotation=0)
axes[0].set_ylabel('Count')

# pH distribution
ph_valid = metadata_df[metadata_df['pH'].notna()]
axes[1].hist(ph_valid['pH'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(PH_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold ({PH_THRESHOLD})')
axes[1].set_title('Umbilical Cord pH Distribution')
axes[1].set_xlabel('pH')
axes[1].legend()

# Apgar5 distribution
apgar_valid = metadata_df[metadata_df['Apgar5'].notna()]
axes[2].hist(apgar_valid['Apgar5'], bins=11, range=(-0.5, 10.5), color='coral', edgecolor='white', alpha=0.8)
axes[2].axvline(APGAR5_THRESHOLD - 0.5, color='red', linestyle='--', linewidth=2, label=f'Threshold ({APGAR5_THRESHOLD})')
axes[2].set_title('5-minute Apgar Score Distribution')
axes[2].set_xlabel('Apgar5')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Exploratory Data Analysis: CTG Signals

In [ ]:
def load_ctg_signal(record_id: str, data_dir: str = RAW_DATA_DIR):
    """Load FHR and UC signals from a WFDB record."""
    record_path = os.path.join(data_dir, record_id)
    record = wfdb.rdrecord(record_path)
    fhr = record.p_signal[:, 0]
    uc = record.p_signal[:, 1]
    fs = record.fs
    return fhr, uc, fs

# Check recording lengths
lengths = []
for rid in tqdm(metadata_df['record_id'].values, desc='Checking lengths'):
    try:
        fhr, uc, fs = load_ctg_signal(rid)
        lengths.append({'record_id': rid, 'n_samples': len(fhr), 'duration_min': len(fhr) / (fs * 60)})
    except Exception as e:
        print(f'Error loading {rid}: {e}')

lengths_df = pd.DataFrame(lengths)
print(f'\nRecording durations (minutes):')
print(lengths_df['duration_min'].describe())

In [ ]:
# Plot example CTG traces: one normal, one distressed
normal_ids = metadata_df[metadata_df['distress'] == 0]['record_id'].values
distress_ids = metadata_df[metadata_df['distress'] == 1]['record_id'].values

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

for col, (label, rec_id) in enumerate([('Normal', normal_ids[0]), ('Distressed', distress_ids[0])]):
    fhr, uc, fs = load_ctg_signal(rec_id)
    time_min = np.arange(len(fhr)) / (fs * 60)
    
    # Last 30 minutes for visibility
    last_n = min(len(fhr), 30 * 60 * fs)
    
    axes[0, col].plot(time_min[-last_n:], fhr[-last_n:], linewidth=0.5, color='darkblue')
    axes[0, col].set_title(f'{label} — Record {rec_id} — FHR')
    axes[0, col].set_ylabel('FHR (bpm)')
    axes[0, col].set_ylim(50, 210)
    
    axes[1, col].plot(time_min[-last_n:], uc[-last_n:], linewidth=0.5, color='darkred')
    axes[1, col].set_title(f'{label} — Record {rec_id} — UC')
    axes[1, col].set_ylabel('UC')
    axes[1, col].set_xlabel('Time (minutes)')

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'example_ctg_traces.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Signal quality analysis: how much missing data (FHR == 0) per recording?
missing_pcts = []
for rid in tqdm(metadata_df['record_id'].values, desc='Signal quality'):
    fhr, _, _ = load_ctg_signal(rid)
    pct_missing = (fhr == 0).sum() / len(fhr) * 100
    missing_pcts.append(pct_missing)

metadata_df['fhr_missing_pct'] = missing_pcts

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(metadata_df['fhr_missing_pct'], bins=50, color='steelblue', edgecolor='white')
ax.set_title('FHR Signal Loss per Recording (% of samples = 0)')
ax.set_xlabel('% Missing')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Median missing: {np.median(missing_pcts):.1f}%')
print(f'Max missing: {np.max(missing_pcts):.1f}%')
print(f'Records with >50% missing: {sum(1 for p in missing_pcts if p > 50)}')

## 5. Preprocessing Pipeline

Key decisions:
1. **Handle missing FHR**: values of 0 = signal loss → interpolate  
2. **Clip** to physiological range: FHR [50, 210] bpm, UC [0, 100]  
3. **Take last 20 minutes** (4800 samples @ 4 Hz) — the period closest to delivery is most clinically informative  
4. **Z-normalise** using training set statistics  
5. **Exclude** recordings with >50% missing FHR data

In [ ]:
SEQ_LENGTH = 4800  # 20 minutes at 4 Hz
MAX_MISSING_PCT = 50.0  # exclude if more than 50% of FHR is missing

# Filter out poor-quality recordings
quality_mask = metadata_df['fhr_missing_pct'] <= MAX_MISSING_PCT
print(f'Recordings passing quality filter: {quality_mask.sum()}/{len(metadata_df)}')
metadata_df = metadata_df[quality_mask].copy().reset_index(drop=True)

In [ ]:
def preprocess_signal(fhr, uc, target_length=SEQ_LENGTH):
    """
    Clean and prepare a CTG signal pair.
    Does NOT normalise — normalisation is done after split to avoid leakage.
    """
    fhr = fhr.copy().astype(np.float64)
    uc = uc.copy().astype(np.float64)
    
    # Handle missing FHR (0 = signal loss)
    fhr[fhr == 0] = np.nan
    fhr[(fhr < 50) | (fhr > 210)] = np.nan
    
    # Interpolate
    if np.any(np.isnan(fhr)):
        nans = np.isnan(fhr)
        if not np.all(nans):
            x = np.arange(len(fhr))
            fhr[nans] = np.interp(x[nans], x[~nans], fhr[~nans])
        else:
            fhr[:] = 140.0  # baseline fallback
    
    # Clip
    fhr = np.clip(fhr, 50, 210)
    uc = np.clip(uc, 0, 100)
    
    # Take last target_length samples (closest to delivery)
    if len(fhr) >= target_length:
        fhr = fhr[-target_length:]
        uc = uc[-target_length:]
    else:
        pad_len = target_length - len(fhr)
        fhr = np.pad(fhr, (pad_len, 0), mode='constant', constant_values=0)
        uc = np.pad(uc, (pad_len, 0), mode='constant', constant_values=0)
    
    return np.stack([fhr, uc], axis=0)  # (2, target_length)

In [ ]:
# Process all recordings
all_signals = []
all_labels = []
valid_ids = []

for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc='Preprocessing'):
    rid = row['record_id']
    try:
        fhr, uc, fs = load_ctg_signal(rid)
        signal = preprocess_signal(fhr, uc)
        all_signals.append(signal)
        all_labels.append(row['distress'])
        valid_ids.append(rid)
    except Exception as e:
        print(f'Error processing {rid}: {e}')

X = np.array(all_signals)   # (N, 2, SEQ_LENGTH)
y = np.array(all_labels)    # (N,)

print(f'\nProcessed dataset: X.shape = {X.shape}, y.shape = {y.shape}')
print(f'Class 0 (normal): {(y == 0).sum()}, Class 1 (distressed): {(y == 1).sum()}')

## 6. Train / Validation / Test Split and Normalisation

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
X_trainval, X_test, y_trainval, y_test, ids_trainval, ids_test = train_test_split(
    X, y, valid_ids, test_size=0.15, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_trainval, y_trainval, ids_trainval, test_size=0.176, random_state=SEED, stratify=y_trainval
    # 0.176 of 85% ≈ 15% of total
)

print(f'Train: {len(X_train)} (distress: {y_train.sum()})')
print(f'Val:   {len(X_val)} (distress: {y_val.sum()})')
print(f'Test:  {len(X_test)} (distress: {y_test.sum()})')

In [ ]:
# Compute normalisation statistics from TRAINING set only (no leakage)
fhr_train = X_train[:, 0, :].flatten()
uc_train = X_train[:, 1, :].flatten()

# Remove padded zeros from stat calculation
fhr_nonzero = fhr_train[fhr_train != 0]
uc_nonzero = uc_train[uc_train != 0]

scaler_stats = {
    'fhr_mean': float(np.mean(fhr_nonzero)),
    'fhr_std': float(np.std(fhr_nonzero)),
    'uc_mean': float(np.mean(uc_nonzero)),
    'uc_std': float(np.std(uc_nonzero)),
}

print('Scaler statistics (from training set):')
for k, v in scaler_stats.items():
    print(f'  {k}: {v:.4f}')

# Save for inference
with open(os.path.join(ARTIFACTS_DIR, 'scaler_stats.json'), 'w') as f:
    json.dump(scaler_stats, f, indent=2)

In [ ]:
def normalise(X, stats):
    """Z-normalise using precomputed stats. Operates in-place on a copy."""
    X = X.copy()
    X[:, 0, :] = (X[:, 0, :] - stats['fhr_mean']) / (stats['fhr_std'] + 1e-8)
    X[:, 1, :] = (X[:, 1, :] - stats['uc_mean']) / (stats['uc_std'] + 1e-8)
    return X

X_train_norm = normalise(X_train, scaler_stats)
X_val_norm = normalise(X_val, scaler_stats)
X_test_norm = normalise(X_test, scaler_stats)

In [ ]:
# Save processed data for sharing
np.savez_compressed(
    os.path.join(PROCESSED_DIR, 'train.npz'),
    X=X_train_norm, y=y_train, ids=ids_train
)
np.savez_compressed(
    os.path.join(PROCESSED_DIR, 'val.npz'),
    X=X_val_norm, y=y_val, ids=ids_val
)
np.savez_compressed(
    os.path.join(PROCESSED_DIR, 'test.npz'),
    X=X_test_norm, y=y_test, ids=ids_test
)

# Save a small sample for inference demo
np.savez_compressed(
    os.path.join(PROCESSED_DIR, 'sample_inference.npz'),
    X=X_test_norm[:5], y=y_test[:5], ids=ids_test[:5]
)

print('Saved processed data to data/processed/')

## 7. PyTorch Dataset and DataLoader

In [ ]:
class CTGDataset(Dataset):
    """PyTorch Dataset for CTG signal classification."""
    
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Handle class imbalance with weighted sampling
class_counts = np.bincount(y_train.astype(int))
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_train.astype(int)]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(y_train),
    replacement=True
)

BATCH_SIZE = 32

train_dataset = CTGDataset(X_train_norm, y_train)
val_dataset = CTGDataset(X_val_norm, y_val)
test_dataset = CTGDataset(X_test_norm, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

## 8. Model Training

In [ ]:
# Instantiate model
model = FetalDistressModel(
    input_channels=2,
    cnn_channels=[32, 64, 128],
    lstm_hidden=64,
    lstm_layers=2,
    dropout=0.3,
    fc_hidden=64,
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
# Loss function with class weighting
pos_weight = torch.tensor([class_counts[0] / class_counts[1]], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer and scheduler
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

print(f'Positive class weight: {pos_weight.item():.2f}')

In [ ]:
NUM_EPOCHS = 50
PATIENCE = 10  # early stopping patience

train_losses = []
val_losses = []
val_aucs = []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item() * X_batch.size(0)
    
    train_loss = epoch_loss / len(train_dataset)
    train_losses.append(train_loss)
    
    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_probs = []
    val_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            
            probs = torch.sigmoid(logits).cpu().numpy()
            val_probs.extend(probs.flatten())
            val_targets.extend(y_batch.cpu().numpy().flatten())
    
    val_loss /= len(val_dataset)
    val_losses.append(val_loss)
    
    # AUC
    try:
        val_auc = roc_auc_score(val_targets, val_probs)
    except ValueError:
        val_auc = 0.0
    val_aucs.append(val_auc)
    
    # Scheduler step
    scheduler.step(val_loss)
    
    # Early stopping & checkpointing
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        model.save(ARTIFACTS_DIR)
        best_epoch = epoch
    else:
        patience_counter += 1
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val AUC: {val_auc:.4f} | '
              f'LR: {optimizer.param_groups[0]["lr"]:.2e}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}')
        break

print(f'\nTraining complete. Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.axvline(best_epoch - 1, color='gray', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()

ax2.plot(val_aucs, label='Val AUC-ROC', color='green')
ax2.axvline(best_epoch - 1, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Validation AUC-ROC')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Evaluation on Test Set

In [ ]:
# Load best model
best_model = FetalDistressModel.load(ARTIFACTS_DIR, device=device)

# Predict on test set
test_probs = []
test_targets = []

best_model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        probs = best_model.predict_proba(X_batch).cpu().numpy()
        test_probs.extend(probs.flatten())
        test_targets.extend(y_batch.numpy().flatten())

test_probs = np.array(test_probs)
test_targets = np.array(test_targets)
test_preds = (test_probs >= 0.5).astype(int)

In [ ]:
# --- Metrics ---
print('=' * 60)
print('TEST SET EVALUATION')
print('=' * 60)

# AUC-ROC
auc_roc = roc_auc_score(test_targets, test_probs)
print(f'\nAUC-ROC:   {auc_roc:.4f}')

# Average Precision (AUC-PR)
avg_prec = average_precision_score(test_targets, test_probs)
print(f'AUC-PR:    {avg_prec:.4f}')

# F1 Score
f1 = f1_score(test_targets, test_preds)
print(f'F1 Score:  {f1:.4f}')

# MCC (Matthews Correlation Coefficient)
mcc = matthews_corrcoef(test_targets, test_preds)
print(f'MCC:       {mcc:.4f}')

# Classification Report
print(f'\nClassification Report (threshold=0.5):')
print(classification_report(test_targets, test_preds, target_names=['Normal', 'Distressed']))

# Confusion Matrix
cm = confusion_matrix(test_targets, test_preds)
print(f'Confusion Matrix:')
print(cm)

In [ ]:
# --- Visualise evaluation ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. ROC Curve
fpr, tpr, roc_thresholds = roc_curve(test_targets, test_probs)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc_roc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('ROC Curve')
axes[0].legend()

# 2. Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(test_targets, test_probs)
axes[1].plot(recall, precision, color='green', lw=2, label=f'AP = {avg_prec:.3f}')
baseline = test_targets.sum() / len(test_targets)
axes[1].axhline(baseline, color='gray', linestyle='--', label=f'Baseline = {baseline:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

# 3. Confusion Matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Normal', 'Distressed'], yticklabels=['Normal', 'Distressed'])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'evaluation_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Threshold Analysis

In a clinical setting, **sensitivity (recall) is paramount** — missing a distressed baby is far worse than a false alarm. Let's find the optimal threshold for different objectives.

In [ ]:
# Find thresholds for different sensitivity targets
from sklearn.metrics import recall_score, precision_score

thresholds = np.arange(0.1, 0.91, 0.05)
threshold_results = []

for t in thresholds:
    preds = (test_probs >= t).astype(int)
    sens = recall_score(test_targets, preds, zero_division=0)
    spec = recall_score(1 - test_targets, 1 - preds, zero_division=0)
    prec = precision_score(test_targets, preds, zero_division=0)
    f1_t = f1_score(test_targets, preds, zero_division=0)
    threshold_results.append({'threshold': t, 'sensitivity': sens, 'specificity': spec,
                              'precision': prec, 'f1': f1_t})

thresh_df = pd.DataFrame(threshold_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['threshold'], thresh_df['sensitivity'], 'r-o', label='Sensitivity', markersize=4)
ax.plot(thresh_df['threshold'], thresh_df['specificity'], 'b-o', label='Specificity', markersize=4)
ax.plot(thresh_df['threshold'], thresh_df['f1'], 'g-o', label='F1 Score', markersize=4)
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Sensitivity, Specificity, and F1 vs. Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'threshold_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(thresh_df.to_string(index=False))

## 11. Save Final Test Predictions

In [ ]:
# Save test predictions for review
test_results_df = pd.DataFrame({
    'record_id': ids_test,
    'true_label': test_targets.astype(int),
    'distress_prob': np.round(test_probs, 4),
    'predicted': test_preds,
})
test_results_df.to_csv(os.path.join(ARTIFACTS_DIR, 'test_predictions.csv'), index=False)
print('Saved test predictions to artifacts/test_predictions.csv')
test_results_df.head(10)

## 12. Summary

### What was built
- A **1D CNN-LSTM** model that takes 20 minutes of CTG signals (FHR + UC) and predicts fetal distress.
- **Distress defined as**: umbilical cord pH < 7.15 (with Apgar5 < 7 as fallback).
- The model handles signal quality issues (missing FHR, sensor dropout) through interpolation.

### Key observations
- The dataset has significant class imbalance (distress is the minority class), addressed via weighted sampling and class-weighted loss.
- AUC-ROC and AUC-PR are more informative than accuracy for this imbalanced problem.
- The threshold can be tuned to prioritise sensitivity (clinical safety) over specificity.

### Limitations
- Small dataset (552 recordings) limits generalisability.
- We only use the last 20 minutes — the model does not see the full labour trajectory.
- No feature engineering from clinical metadata (gestational age, parity, etc.) — could improve performance.
- The model is trained end-to-end on fixed windows; a sliding-window approach would be more realistic for real-time monitoring.

In [ ]:
print('Done! All artifacts saved.')
print(f'\nProject artifacts:')
for f in sorted(os.listdir(ARTIFACTS_DIR)):
    size = os.path.getsize(os.path.join(ARTIFACTS_DIR, f))
    print(f'  {f} ({size / 1024:.1f} KB)')